# Notebook 02 — Machine Learning Não Supervisionado

## Perfis de denúncias relacionadas ao tráfico de pessoas no Disque 100

Este notebook constrói a unidade analítica no nível da denúncia, preserva todas as violações concomitantes nas denúncias que contêm tráfico de pessoas e compara representações hierárquicas para agrupamento não supervisionado.

### Fluxo metodológico

1. carregar os dados preparados no Notebook 01;
2. identificar denúncias com pelo menos uma ocorrência de tráfico;
3. recuperar todas as ocorrências dessas denúncias;
4. construir matrizes binárias denúncia × classe, subclasse e espécie;
5. avaliar suporte, prevalência, variância e esparsidade;
6. comparar representações;
7. aplicar modelos de referência e modelos principais;
8. avaliar estabilidade, separação e interpretabilidade;
9. caracterizar os clusters pelas espécies e demais atributos.

> **Importante:** os algoritmos não recebem o termo usado para selecionar o subconjunto como uma variável de segmentação quando esse termo é constante ou quase constante entre todas as denúncias.

## 1. Configuração e reprodutibilidade

In [ ]:
import importlib.util
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from IPython.display import display
from scipy import sparse
from scipy.spatial.distance import pdist, squareform
from sklearn.cluster import AgglomerativeClustering, KMeans
from sklearn.decomposition import TruncatedSVD
from sklearn.metrics import (
    silhouette_score,
    davies_bouldin_score,
    adjusted_rand_score,
    pairwise_distances,
)
from sklearn.preprocessing import normalize

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
pd.set_option("display.max_columns", 120)
pd.set_option("display.max_colwidth", 120)

COL_DENUNCIA = "hash"
COL_VIOLACAO = "violacao"
COL_CLASSE = "violacao_classe"
COL_SUBCLASSE = "violacao_subclasse"
COL_ESPECIE = "violacao_especie_norm"
COL_ABRANGENCIA = "violacao_abrangencia_norm"

TERMO_TRAFICO = r"TRÁFICO|TRAFICO"
SUPORTE_MINIMO = 0.01
SUPORTE_MAXIMO = 0.99
MIN_OCORRENCIAS = 5

print("Versões")
print("pandas:", pd.__version__)
print("numpy:", np.__version__)
print("umap disponível:", importlib.util.find_spec("umap") is not None)
print("hdbscan disponível:", importlib.util.find_spec("hdbscan") is not None)

## 2. Carregamento dos dados preparados

A célula aceita duas formas de continuidade:

- `df_norm` já existente na sessão após a execução do Notebook 01; ou
- arquivo Parquet exportado pelo Notebook 01.

Ajuste `CAMINHO_DADOS` somente se necessário.

In [ ]:
CAMINHO_DADOS = Path("dados_disque100_taxonomia_expandida.parquet")

if "df_norm" in globals():
    dados = df_norm.copy()
    origem_dados = "DataFrame df_norm em memória"
elif CAMINHO_DADOS.exists():
    dados = pd.read_parquet(CAMINHO_DADOS)
    origem_dados = str(CAMINHO_DADOS)
else:
    raise FileNotFoundError(
        "Execute o Notebook 01 ou exporte `df_norm` para "
        f"{CAMINHO_DADOS.resolve()}."
    )

required = [COL_DENUNCIA, COL_VIOLACAO, COL_CLASSE, COL_SUBCLASSE, COL_ESPECIE]
missing = [column for column in required if column not in dados.columns]
assert not missing, f"Colunas obrigatórias ausentes: {missing}"

print("Origem:", origem_dados)
print("Dimensão:", dados.shape)
print("Denúncias distintas:", dados[COL_DENUNCIA].nunique(dropna=True))
display(dados[required + ([COL_ABRANGENCIA] if COL_ABRANGENCIA in dados.columns else [])].head())

## 3. Auditoria da chave da denúncia

In [ ]:
auditoria_chave = pd.Series({
    "linhas": len(dados),
    "hash_ausente": int(dados[COL_DENUNCIA].isna().sum()),
    "denuncias_distintas": int(dados[COL_DENUNCIA].nunique(dropna=True)),
    "linhas_duplicadas_completas": int(dados.duplicated().sum()),
    "pares_hash_violacao_duplicados": int(
        dados.duplicated(subset=[COL_DENUNCIA, COL_VIOLACAO]).sum()
    ),
})
display(auditoria_chave.to_frame("valor"))

# Registros sem hash não podem formar corretamente a unidade denúncia.
dados_modelagem = dados.loc[dados[COL_DENUNCIA].notna()].copy()

## 4. Seleção do subconjunto relacionado ao tráfico

Primeiro são identificados os `hash` que contêm tráfico na espécie normalizada. Em seguida, todas as linhas desses `hash` são recuperadas, incluindo violações físicas, psíquicas, sexuais, laborais, retenção de documentos e outras violações concomitantes.

In [ ]:
mask_trafico = (
    dados_modelagem[COL_ESPECIE]
    .astype("string")
    .str.contains(TERMO_TRAFICO, case=False, regex=True, na=False)
)

hashes_trafico = dados_modelagem.loc[mask_trafico, COL_DENUNCIA].drop_duplicates()
subconjunto = dados_modelagem.loc[
    dados_modelagem[COL_DENUNCIA].isin(hashes_trafico)
].copy()

resumo_selecao = pd.Series({
    "linhas_base_preparada": len(dados_modelagem),
    "denuncias_base_preparada": dados_modelagem[COL_DENUNCIA].nunique(),
    "linhas_que_mencionam_trafico": int(mask_trafico.sum()),
    "denuncias_com_trafico": hashes_trafico.nunique(),
    "linhas_recuperadas_dessas_denuncias": len(subconjunto),
    "violacoes_distintas_no_subconjunto": subconjunto[COL_ESPECIE].nunique(),
})
display(resumo_selecao.to_frame("valor"))

In [ ]:
# Validação: toda denúncia selecionada deve conter ao menos uma ocorrência de tráfico.
validacao_selecao = (
    subconjunto.assign(_trafico=subconjunto[COL_ESPECIE].astype("string").str.contains(
        TERMO_TRAFICO, case=False, regex=True, na=False
    ))
    .groupby(COL_DENUNCIA)["_trafico"]
    .any()
)

assert validacao_selecao.all(), "Há denúncia selecionada sem ocorrência de tráfico."
print("Seleção validada:", validacao_selecao.all())
print("Denúncias validadas:", len(validacao_selecao))

## 5. Descrição da unidade analítica

Esta etapa mede a quantidade de ocorrências e de categorias distintas por denúncia antes da codificação.

In [ ]:
resumo_denuncia = (
    subconjunto.groupby(COL_DENUNCIA)
    .agg(
        ocorrencias=(COL_VIOLACAO, "size"),
        classes=(COL_CLASSE, "nunique"),
        subclasses=(COL_SUBCLASSE, "nunique"),
        especies=(COL_ESPECIE, "nunique"),
    )
)
display(resumo_denuncia.describe(percentiles=[0.25, 0.50, 0.75, 0.90, 0.95, 0.99]).T)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for ax, column, title in zip(
    axes,
    ["classes", "subclasses", "especies"],
    ["Classes por denúncia", "Subclasses por denúncia", "Espécies por denúncia"],
):
    ax.hist(resumo_denuncia[column], bins="auto")
    ax.set_title(title)
    ax.set_xlabel("Quantidade")
    ax.set_ylabel("Denúncias")
plt.tight_layout()
plt.show()

## 6. Construção das matrizes binárias

In [ ]:
def build_binary_matrix(data: pd.DataFrame, id_column: str, category_column: str) -> pd.DataFrame:
    matrix = pd.crosstab(data[id_column], data[category_column])
    matrix = matrix.gt(0).astype("uint8")
    matrix.index.name = id_column
    return matrix

matriz_classe = build_binary_matrix(subconjunto, COL_DENUNCIA, COL_CLASSE)
matriz_subclasse = build_binary_matrix(subconjunto, COL_DENUNCIA, COL_SUBCLASSE)
matriz_especie = build_binary_matrix(subconjunto, COL_DENUNCIA, COL_ESPECIE)

matrizes_brutas = {
    "classe": matriz_classe,
    "subclasse": matriz_subclasse,
    "especie": matriz_especie,
}

pd.DataFrame({
    nome: {
        "denuncias": matriz.shape[0],
        "atributos": matriz.shape[1],
        "valores_1": int(matriz.to_numpy().sum()),
        "densidade": float(matriz.to_numpy().mean()),
        "esparsidade": 1 - float(matriz.to_numpy().mean()),
    }
    for nome, matriz in matrizes_brutas.items()
}).T

## 7. Suporte, prevalência e capacidade discriminatória

O suporte é calculado como a proporção de denúncias que possui cada categoria. Variáveis presentes em praticamente todas as denúncias ou em pouquíssimas denúncias tendem a ter baixa capacidade de separar agrupamentos.

Os valores padrão são:

- suporte mínimo proporcional: 1%;
- mínimo absoluto: 5 denúncias;
- suporte máximo: 99%.

Esses parâmetros devem ser analisados em conjunto, e não aplicados como regra universal.

In [ ]:
def feature_diagnostics(matrix: pd.DataFrame) -> pd.DataFrame:
    count = matrix.sum(axis=0)
    support = matrix.mean(axis=0)
    variance = support * (1 - support)
    result = pd.DataFrame({
        "contagem_denuncias": count,
        "suporte": support,
        "variancia_bernoulli": variance,
    })
    return result.sort_values("suporte", ascending=False)

diagnosticos = {
    nome: feature_diagnostics(matriz)
    for nome, matriz in matrizes_brutas.items()
}

for nome, diagnostico in diagnosticos.items():
    print(f"
{nome.upper()}")
    display(diagnostico.head(15))

In [ ]:
def filter_features(
    matrix: pd.DataFrame,
    min_support: float = SUPORTE_MINIMO,
    max_support: float = SUPORTE_MAXIMO,
    min_count: int = MIN_OCORRENCIAS,
):
    diagnostics = feature_diagnostics(matrix)
    selected = diagnostics.index[
        diagnostics["contagem_denuncias"].ge(min_count)
        & diagnostics["suporte"].ge(min_support)
        & diagnostics["suporte"].le(max_support)
    ]
    return matrix.loc[:, selected].copy(), diagnostics

matrizes_filtradas = {}
relatorio_filtro = []

for nome, matriz in matrizes_brutas.items():
    filtrada, diagnostico = filter_features(matriz)
    matrizes_filtradas[nome] = filtrada
    relatorio_filtro.append({
        "nivel": nome,
        "atributos_antes": matriz.shape[1],
        "atributos_depois": filtrada.shape[1],
        "removidos": matriz.shape[1] - filtrada.shape[1],
        "densidade_depois": filtrada.to_numpy().mean() if filtrada.shape[1] else np.nan,
    })

display(pd.DataFrame(relatorio_filtro))

## 8. Tratamento do atributo usado na seleção

Como todas as denúncias foram selecionadas por conter tráfico, uma coluna equivalente a `TRÁFICO DE PESSOAS` pode ser constante. A coluna deve permanecer disponível para auditoria e interpretação, mas não deve entrar no clustering quando tiver variância zero ou prevalência extrema.

In [ ]:
trafico_features = [
    column for column in matriz_especie.columns
    if pd.Series([column], dtype="string").str.contains(
        TERMO_TRAFICO, case=False, regex=True, na=False
    ).iloc[0]
]

print("Atributos de espécie associados ao termo de seleção:")
for column in trafico_features:
    print(
        f"- {column}: "
        f"{int(matriz_especie[column].sum())} denúncias "
        f"({matriz_especie[column].mean():.1%})"
    )

## 9. Comparação das representações

In [ ]:
comparacao_representacoes = []
for nome, matriz in matrizes_filtradas.items():
    if matriz.shape[1] == 0:
        continue
    row_sums = matriz.sum(axis=1)
    duplicate_rate = matriz.duplicated().mean()
    comparacao_representacoes.append({
        "nivel": nome,
        "denuncias": matriz.shape[0],
        "atributos": matriz.shape[1],
        "media_atributos_ativos": row_sums.mean(),
        "mediana_atributos_ativos": row_sums.median(),
        "densidade": matriz.to_numpy().mean(),
        "esparsidade": 1 - matriz.to_numpy().mean(),
        "proporcao_perfis_duplicados": duplicate_rate,
    })

display(pd.DataFrame(comparacao_representacoes).set_index("nivel"))

## 10. Distância de Jaccard e agrupamento hierárquico

A distância de Jaccard compara conjuntos de violações presentes em cada denúncia e ignora ausências compartilhadas. Isso é apropriado para matrizes binárias esparsas.

Para evitar consumo excessivo de memória, a matriz completa de distâncias só deve ser criada quando o número de denúncias for administrável. O limite padrão abaixo é 5.000 denúncias.

In [ ]:
NIVEL_PRINCIPAL = "subclasse"
X_binary = matrizes_filtradas[NIVEL_PRINCIPAL].copy()
assert X_binary.shape[1] >= 2, "A representação selecionada possui poucos atributos."

MAX_JACCARD_FULL = 5000
if len(X_binary) <= MAX_JACCARD_FULL:
    distancia_jaccard = pairwise_distances(X_binary, metric="jaccard")
    print("Matriz Jaccard criada:", distancia_jaccard.shape)
else:
    distancia_jaccard = None
    print(
        f"Matriz Jaccard completa não criada: {len(X_binary):,} denúncias excedem "
        f"o limite de {MAX_JACCARD_FULL:,}. Use amostragem ou HDBSCAN com representação reduzida."
    )

In [ ]:
def evaluate_precomputed_clustering(distance_matrix, labels):
    valid = labels >= 0
    n_clusters = len(set(labels[valid]))
    if valid.sum() < 3 or n_clusters < 2:
        return np.nan
    return silhouette_score(
        distance_matrix[np.ix_(valid, valid)],
        labels[valid],
        metric="precomputed",
    )

resultados_aglomerativo = []
modelos_aglomerativos = {}

if distancia_jaccard is not None:
    max_k = min(10, len(X_binary) - 1)
    for k in range(2, max_k + 1):
        model = AgglomerativeClustering(
            n_clusters=k,
            metric="precomputed",
            linkage="average",
        )
        labels = model.fit_predict(distancia_jaccard)
        score = evaluate_precomputed_clustering(distancia_jaccard, labels)
        resultados_aglomerativo.append({"k": k, "silhouette_jaccard": score})
        modelos_aglomerativos[k] = (model, labels)

    display(pd.DataFrame(resultados_aglomerativo).sort_values("silhouette_jaccard", ascending=False))

## 11. Baseline: Truncated SVD + K-Means

K-Means não é o modelo principal para dados binários esparsos, mas funciona como baseline após redução com Truncated SVD. A comparação ajuda a verificar se a estrutura encontrada depende exclusivamente do algoritmo principal.

In [ ]:
n_components = min(20, X_binary.shape[1] - 1, X_binary.shape[0] - 1)
assert n_components >= 2, "Dados insuficientes para Truncated SVD."

svd = TruncatedSVD(n_components=n_components, random_state=RANDOM_STATE)
X_svd = svd.fit_transform(sparse.csr_matrix(X_binary))
X_svd_norm = normalize(X_svd)

print("Componentes:", n_components)
print("Variância explicada acumulada:", svd.explained_variance_ratio_.sum())

In [ ]:
resultados_kmeans = []
modelos_kmeans = {}
max_k = min(10, len(X_binary) - 1)

for k in range(2, max_k + 1):
    model = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=20)
    labels = model.fit_predict(X_svd_norm)
    resultados_kmeans.append({
        "k": k,
        "silhouette_cosine": silhouette_score(X_svd_norm, labels, metric="cosine"),
        "davies_bouldin": davies_bouldin_score(X_svd_norm, labels),
        "inercia": model.inertia_,
    })
    modelos_kmeans[k] = (model, labels)

resultados_kmeans_df = pd.DataFrame(resultados_kmeans)
display(resultados_kmeans_df.sort_values("silhouette_cosine", ascending=False))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
axes[0].plot(resultados_kmeans_df["k"], resultados_kmeans_df["silhouette_cosine"], marker="o")
axes[0].set_title("Silhouette com distância cosseno")
axes[1].plot(resultados_kmeans_df["k"], resultados_kmeans_df["davies_bouldin"], marker="o")
axes[1].set_title("Davies-Bouldin: menor é melhor")
axes[2].plot(resultados_kmeans_df["k"], resultados_kmeans_df["inercia"], marker="o")
axes[2].set_title("Inércia")
for ax in axes:
    ax.set_xlabel("Número de clusters")
plt.tight_layout()
plt.show()

## 12. Modelo principal opcional: UMAP + HDBSCAN

O bloco é executado somente se `umap-learn` e `hdbscan` estiverem instalados. Para matrizes binárias, UMAP usa distância de Jaccard. O HDBSCAN identifica agrupamentos de densidade variável e pode marcar denúncias como ruído (`cluster = -1`).

Instalação no ambiente do notebook, se necessária:

`%pip install umap-learn hdbscan`

In [ ]:
HAS_UMAP = importlib.util.find_spec("umap") is not None
HAS_HDBSCAN = importlib.util.find_spec("hdbscan") is not None

if HAS_UMAP and HAS_HDBSCAN:
    import umap.umap_ as umap
    import hdbscan

    n_neighbors = min(30, max(2, len(X_binary) - 1))
    reducer = umap.UMAP(
        n_components=10,
        n_neighbors=n_neighbors,
        min_dist=0.0,
        metric="jaccard",
        random_state=RANDOM_STATE,
    )
    X_umap = reducer.fit_transform(X_binary.astype(bool))

    min_cluster_size = max(10, int(np.ceil(0.02 * len(X_binary))))
    min_samples = max(5, min_cluster_size // 2)

    modelo_hdbscan = hdbscan.HDBSCAN(
        min_cluster_size=min_cluster_size,
        min_samples=min_samples,
        metric="euclidean",
        cluster_selection_method="eom",
        prediction_data=True,
    )
    labels_hdbscan = modelo_hdbscan.fit_predict(X_umap)

    print("min_cluster_size:", min_cluster_size)
    print("min_samples:", min_samples)
    print("clusters:", len(set(labels_hdbscan)) - (1 if -1 in labels_hdbscan else 0))
    print("ruído:", np.mean(labels_hdbscan == -1))
else:
    X_umap = None
    labels_hdbscan = None
    print("UMAP + HDBSCAN não executado. Instale `umap-learn` e `hdbscan`.")

In [ ]:
if X_umap is not None:
    valid = labels_hdbscan >= 0
    n_clusters = len(set(labels_hdbscan[valid]))
    silhouette_hdbscan = (
        silhouette_score(X_umap[valid], labels_hdbscan[valid])
        if valid.sum() >= 3 and n_clusters >= 2
        else np.nan
    )
    print("Silhouette sem ruído:", silhouette_hdbscan)

    # UMAP bidimensional usado somente para visualização.
    reducer_2d = umap.UMAP(
        n_components=2,
        n_neighbors=min(30, max(2, len(X_binary) - 1)),
        min_dist=0.1,
        metric="jaccard",
        random_state=RANDOM_STATE,
    )
    X_umap_2d = reducer_2d.fit_transform(X_binary.astype(bool))

    fig, ax = plt.subplots(figsize=(9, 7))
    scatter = ax.scatter(
        X_umap_2d[:, 0], X_umap_2d[:, 1],
        c=labels_hdbscan, s=18, alpha=0.75, cmap="tab20"
    )
    ax.set_title("UMAP das denúncias por cluster HDBSCAN")
    ax.set_xlabel("UMAP 1")
    ax.set_ylabel("UMAP 2")
    plt.tight_layout()
    plt.show()

## 13. Escolha explícita do modelo final

A escolha não deve ser feita apenas pelo maior silhouette. Considere também:

- proporção de ruído;
- tamanho mínimo dos clusters;
- estabilidade sob reamostragem;
- coerência das violações predominantes;
- utilidade substantiva para interpretar perfis de tráfico.

A célula escolhe automaticamente HDBSCAN quando disponível e com pelo menos dois clusters; caso contrário, usa o melhor K-Means por silhouette cosseno. A decisão pode ser substituída após a avaliação.

In [ ]:
if labels_hdbscan is not None and len(set(labels_hdbscan[labels_hdbscan >= 0])) >= 2:
    MODELO_FINAL = "umap_hdbscan"
    labels_final = labels_hdbscan
else:
    melhor_k = int(
        resultados_kmeans_df.sort_values(
            ["silhouette_cosine", "davies_bouldin"],
            ascending=[False, True],
        ).iloc[0]["k"]
    )
    MODELO_FINAL = f"svd_kmeans_k{melhor_k}"
    labels_final = modelos_kmeans[melhor_k][1]

clusters = pd.Series(labels_final, index=X_binary.index, name="cluster")
print("Modelo final provisório:", MODELO_FINAL)
display(clusters.value_counts(dropna=False).sort_index().to_frame("denuncias"))

## 14. Estabilidade por reamostragem

In [ ]:
def bootstrap_kmeans_stability(X, k, iterations=30, sample_fraction=0.80, random_state=42):
    rng = np.random.default_rng(random_state)
    base = KMeans(n_clusters=k, random_state=random_state, n_init=20).fit_predict(X)
    scores = []

    n_sample = max(k + 1, int(np.floor(sample_fraction * len(X))))
    for i in range(iterations):
        idx = np.sort(rng.choice(len(X), size=n_sample, replace=False))
        labels = KMeans(
            n_clusters=k,
            random_state=random_state + i + 1,
            n_init=20,
        ).fit_predict(X[idx])
        scores.append(adjusted_rand_score(base[idx], labels))

    return pd.Series(scores, name="adjusted_rand_index")

if MODELO_FINAL.startswith("svd_kmeans"):
    estabilidade = bootstrap_kmeans_stability(X_svd_norm, melhor_k)
    display(estabilidade.describe().to_frame("resultado"))
else:
    print(
        "Para HDBSCAN, recomenda-se repetir UMAP + HDBSCAN em subamostras e comparar "
        "pares de denúncias com Adjusted Rand Index. A implementação depende da proporção "
        "de ruído e será ajustada após observar o modelo final."
    )

## 15. Perfil dos clusters pelas subclasses

A prevalência é a proporção de denúncias do cluster que contém a categoria. O *lift* compara essa prevalência com a prevalência global. Valores maiores que 1 indicam sobrerrepresentação.

In [ ]:
def cluster_feature_profile(matrix: pd.DataFrame, labels: pd.Series, top_n=12):
    aligned = matrix.loc[labels.index]
    global_prevalence = aligned.mean(axis=0)
    rows = []

    for cluster_id in sorted(labels.unique()):
        member_ids = labels.index[labels.eq(cluster_id)]
        prevalence = aligned.loc[member_ids].mean(axis=0)
        lift = prevalence.div(global_prevalence.replace(0, np.nan))
        table = pd.DataFrame({
            "cluster": cluster_id,
            "atributo": prevalence.index,
            "prevalencia_cluster": prevalence.values,
            "prevalencia_global": global_prevalence.values,
            "lift": lift.values,
            "denuncias_cluster": len(member_ids),
        })
        table = table.sort_values(
            ["lift", "prevalencia_cluster"], ascending=False
        ).head(top_n)
        rows.append(table)

    return pd.concat(rows, ignore_index=True)

perfil_subclasse = cluster_feature_profile(
    matriz_subclasse.loc[clusters.index], clusters, top_n=12
)
display(perfil_subclasse)

## 16. Interpretação dos clusters pelas espécies

In [ ]:
perfil_especie = cluster_feature_profile(
    matriz_especie.loc[clusters.index], clusters, top_n=15
)
display(perfil_especie)

In [ ]:
for cluster_id in sorted(clusters.unique()):
    print(f"
CLUSTER {cluster_id}")
    display(
        perfil_especie.loc[perfil_especie["cluster"].eq(cluster_id), [
            "atributo", "prevalencia_cluster", "prevalencia_global", "lift"
        ]]
    )

## 17. Caracterização externa dos clusters

As variáveis externas não entram obrigatoriamente no clustering. Elas são agregadas por denúncia e usadas para compreender os perfis encontrados. Ajuste `CANDIDATAS_CATEGORICAS` aos nomes existentes no conjunto.

In [ ]:
CANDIDATAS_CATEGORICAS = [
    "ano", "uf", "sexo", "faixa_etaria", "canal_atendimento",
    "relacao_agressor_vitima", "violacao_abrangencia_norm",
]
CATEGORICAS_DISPONIVEIS = [
    column for column in CANDIDATAS_CATEGORICAS if column in subconjunto.columns
]
print("Variáveis externas disponíveis:", CATEGORICAS_DISPONIVEIS)

In [ ]:
def aggregate_mode(series: pd.Series):
    mode = series.dropna().mode()
    return mode.iloc[0] if len(mode) else pd.NA

if CATEGORICAS_DISPONIVEIS:
    metadados_denuncia = (
        subconjunto.groupby(COL_DENUNCIA)[CATEGORICAS_DISPONIVEIS]
        .agg(aggregate_mode)
        .join(clusters)
    )

    for column in CATEGORICAS_DISPONIVEIS:
        print(f"
{column}")
        table = pd.crosstab(
            metadados_denuncia["cluster"],
            metadados_denuncia[column],
            normalize="index",
        )
        display(table)
else:
    metadados_denuncia = clusters.to_frame()
    print("Nenhuma variável externa candidata foi encontrada.")

## 18. Tabela analítica final e exportação

São exportados:

- cluster por denúncia;
- perfis de subclasse;
- perfis de espécie;
- relatório dos filtros de atributos;
- matrizes binárias filtradas, se desejado.

In [ ]:
resultado_denuncias = (
    resumo_denuncia
    .join(clusters, how="inner")
    .reset_index()
)
resultado_denuncias["modelo"] = MODELO_FINAL

relatorio_filtros = pd.DataFrame(relatorio_filtro)

display(resultado_denuncias.head())

In [ ]:
# Descomente após validar a solução final.
# resultado_denuncias.to_csv("clusters_denuncias_trafico.csv", index=False, encoding="utf-8-sig")
# perfil_subclasse.to_csv("perfil_clusters_subclasse.csv", index=False, encoding="utf-8-sig")
# perfil_especie.to_csv("perfil_clusters_especie.csv", index=False, encoding="utf-8-sig")
# relatorio_filtros.to_csv("relatorio_filtro_atributos.csv", index=False, encoding="utf-8-sig")
# matrizes_filtradas["subclasse"].to_parquet("matriz_denuncia_subclasse.parquet")

print("Resultado disponível em `resultado_denuncias`.")

## 19. Checklist de avaliação metodológica

Antes de interpretar os agrupamentos no artigo, registre:

- [ ] número de denúncias selecionadas;
- [ ] número de violações concomitantes preservadas;
- [ ] variáveis removidas por suporte baixo ou prevalência extrema;
- [ ] densidade e esparsidade de cada representação;
- [ ] métrica de distância utilizada;
- [ ] hiperparâmetros dos algoritmos;
- [ ] quantidade e tamanho dos clusters;
- [ ] proporção de ruído, quando aplicável;
- [ ] silhouette e Davies-Bouldin, quando aplicáveis;
- [ ] estabilidade por reamostragem;
- [ ] prevalência e lift das subclasses e espécies em cada cluster;
- [ ] coerência substantiva com a taxonomia do Disque 100;
- [ ] limitações decorrentes de subnotificação, preenchimento administrativo e categorias raras.

## 20. Texto-base para a metodologia

As denúncias relacionadas ao tráfico de pessoas foram identificadas pela presença de pelo menos uma espécie de violação cujo rótulo continha o termo “tráfico”. Após a identificação das chaves das denúncias, todas as violações registradas nas mesmas denúncias foram preservadas, com o objetivo de representar a natureza multidimensional do fenômeno. A unidade de análise foi a denúncia, e as ocorrências foram agregadas em matrizes binárias de presença ou ausência nos níveis de classe, subclasse e espécie da taxonomia.

A seleção dos atributos considerou o número absoluto de denúncias, o suporte proporcional, a prevalência extrema e a variância das variáveis binárias. A representação principal foi definida após a comparação da dimensionalidade, esparsidade e capacidade discriminatória dos níveis taxonômicos. Foram comparados agrupamento hierárquico com distância de Jaccard, K-Means após redução por Truncated SVD e, quando disponíveis, UMAP e HDBSCAN. A avaliação considerou separação interna, proporção de ruído, estabilidade por reamostragem e interpretabilidade substantiva. Os agrupamentos foram caracterizados pelas prevalências e pelos valores de lift das subclasses e espécies de violação, além de variáveis externas agregadas no nível da denúncia.